In [1]:
from dataclasses import dataclass

from dotenv import load_dotenv
from openai.types.responses import response

load_dotenv()

True

In [2]:
from typing import Callable

from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from langchain.chat_models import init_chat_model

large_model = init_chat_model(
    model="groq:llama-3.3-70b-versatile"
)

standard_model = init_chat_model(
    model="groq:llama-3.3-70b-versatile"
)


@wrap_model_call
def state_base_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    # Example: choose model based on message count
    intent=request.runtime.content.intent
    if intent=="Reporting":
        print("Intent is reporting")
        message_count=len(request,response)

        if message_count>10:
            model=large_model
        else:
            model=standard_model
            request=request.override(model=model)
            return handler(request)


In [3]:
from dataclasses import dataclass
from typing import Callable, TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

# ---------------- Models ----------------

large_model = init_chat_model(
    model="groq:openai/gpt-oss-120b")

standard_model = init_chat_model(
    model="groq:llama-3.3-70b-versatile"
)

# ---------------- Custom State ----------------

@dataclass
class MessageIntent:
    intent: str


class AgentState(TypedDict):
    messages: list
    intent: MessageIntent

# ---------------- Middleware ----------------

@wrap_model_call
def state_base_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:

    if request.state["intent"].intent == "Reporting":
        request.model = large_model
    else:
        request.model = standard_model

    return handler(request)

# ---------------- Agent ----------------

agent = create_agent(
    model=standard_model,
    middleware=[state_base_call],
    state_schema=AgentState,
    system_prompt="You are roleplaying as a helpful office intern."
)

# ---------------- Invoke ----------------

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?")
        ],
        "intent": MessageIntent("Reporting"),
    }
)

print(response["messages"][-1].content)
print(response["messages"][-1].response_metadata["model_name"])

C:\Users\Admin new\AppData\Local\Temp\ipykernel_27508\1193881989.py:42: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = large_model


Hey there! 🌱

I gave the office plant a good drink this morning—about a cup of water at the base, just enough to keep the soil moist but not soggy. It looks happy and perkier already!

If you have any specific watering schedule or special care tips (like misting or a favorite spot in the office), just let me know and I’ll make sure to follow them. Happy to keep our green coworker thriving!
openai/gpt-oss-120b


In [3]:
from groq import Groq

client = Groq()

models = client.models.list()

for model in models.data:
    print(model.id)

meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
qwen/qwen3.6-27b
canopylabs/orpheus-arabic-saudi
whisper-large-v3
groq/compound-mini
whisper-large-v3-turbo
groq/compound
openai/gpt-oss-120b
allam-2-7b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
llama-3.1-8b-instant
llama-3.3-70b-versatile


In [4]:
# ============================================================
# CELL 6: Long Conversation — Expect Large Model
# ============================================================
# We inject 11 messages (10 existing + 1 new question).
# 11 > 10 → middleware selects large_model (claude-sonnet-4-5).
#
# Notice we inject prior Human/AI pairs to simulate a long conversation.
# In a real app with checkpointer, this history would accumulate naturally.
# Here we inject it to trigger the threshold in a single test call.

from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")  # 11th message
    ]}
)

print(response["messages"][-1].content)

KeyError: 'intent'

In [5]:
from typing import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage

# ============================================================
# Step 1: Create Models
# ============================================================

large_model = init_chat_model(
    model="groq:llama-3.3-70b-versatile"
)

standard_model = init_chat_model(
    model="groq:llama-3.1-8b-instant"
)

# ============================================================
# Step 2: Middleware
# ============================================================

@wrap_model_call
def state_base_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:

    # Use large model if conversation has more than 10 messages
    if len(request.state["messages"]) > 10:
        print("Using Large Model")
        request.model = large_model
    else:
        print("Using Standard Model")
        request.model = standard_model

    return handler(request)

# ============================================================
# Step 3: Create Agent
# ============================================================

agent = create_agent(
    model=standard_model,
    middleware=[state_base_call],
    system_prompt="You are a helpful office intern."
)

# ============================================================
# Step 4: Short Conversation
# ============================================================

print("========== SHORT CONVERSATION ==========")

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?")
        ]
    }
)

print(response["messages"][-1].content)
print(response["messages"][-1].response_metadata["model_name"])

# ============================================================
# Step 5: Long Conversation
# ============================================================

print("\n========== LONG CONVERSATION ==========")

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?"),
            AIMessage(content="Yes, I watered it this morning."),
            HumanMessage(content="Has it grown much this week?"),
            AIMessage(content="Yes, two new leaves have appeared."),
            HumanMessage(content="Are the leaves still yellow?"),
            AIMessage(content="Only slightly on the edges."),
            HumanMessage(content="Did you rotate the pot?"),
            AIMessage(content="Yes, I rotated it toward the sunlight."),
            HumanMessage(content="How often should we fertilize it?"),
            AIMessage(content="Once every two weeks."),
            HumanMessage(content="When should we replace the pot?")
        ]
    }
)

print(response["messages"][-1].content)
print(response["messages"][-1].response_metadata["model_name"])

========== SHORT CONVERSATION ==========
Using Standard Model


C:\Users\Admin new\AppData\Local\Temp\ipykernel_27508\3868404936.py:40: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = standard_model


I did check on the office plant earlier, but I was supposed to water it. Unfortunately, I misplaced the watering schedule. Let me check with our office manager to see if they know when the last watering was and when the next one is due.
llama-3.1-8b-instant

========== LONG CONVERSATION ==========
Using Large Model


C:\Users\Admin new\AppData\Local\Temp\ipykernel_27508\3868404936.py:37: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = large_model


The manager mentioned that we should replace it when the roots start growing out of the pot, probably in about 3-4 months.
llama-3.3-70b-versatile
